### Simulación #1: Descargar y almacenar una copia de datos de los procesos judiciales de Colombia

**Paso 1.** Configurar las rutas de Python para poder hacer uso de la implementación del *Piloto de Framework*

In [1]:
import sys
import os.path
from pathlib import Path
sys.path.append(str(Path(os.path.abspath('')).parent))
import logging
logger = logging.getLogger("root")
logger.setLevel(logging.ERROR)
#logger.setLevel(logging.WARNING)

**Paso 2.** Importar las librerías que conforman la implementación del *Piloto de Framework*

In [2]:
import re
import pandas as pd
from datetime import datetime
from sodapy import Socrata

from framework.integraciones.piloto_framework import piloto_framework
from framework.modelos.repositorio_datos import EstadoEjecucion, TipoCarga
import dotenv
import sqlalchemy
dotenv.load_dotenv()

True

**Paso 3.** Configurar las variables básicas necesarias para el funcionamiento del *Piloto de Framework*
* Definir la variable **id_proceso** con el identificador del proceso a ejecutar. Equivale a *PROCESO.ID_PROCESO* del *Repositorio de datos*
* Definir la variable **parametros** con el listado de identificadores de los parámetros a usar. Equivale a *PARAMETRO.ID_PARAMETRO* del *Repositorio de datos*

In [3]:
id_proceso = 'a0b51bd8-ad38-4cf1-b587-eb2ac1d9cf34'
parametros = ["tabla_destino_p1","url_fuente_p1","limite_fuente_p1","id_fuente_p1"]

**Paso 4.** Ejecutar los métodos necesarios para realizar la configuración inicial del proceso y los parámetros

In [4]:
framework = piloto_framework()
framework.configurar_proceso(id_proceso)
framework.configurar_parametros(parametros)
framework.ver_datos_proceso()
framework.ver_parametros()

╭───────────────────────────────────────────── Detalles del proceso ──────────────────────────────────────────────╮
│ {                                                                                                               │
│   "descripcion_proceso": "Proceso que descarga los procesos judiciales de Colombia desde el portal de Datos Abi │
│   "id_proceso": "a0b51bd8-ad38-4cf1-b587-eb2ac1d9cf34",                                                         │
│   "id_proceso_padre": null,                                                                                     │
│   "fecha_inicio_extraccion": null,                                                                              │
│   "email_responsable_proceso": "sirghoro@gmail.com",                                                            │
│   "estado_ultima_ejecucion": "Correcto",                                                                        │
│   "usuario_responsable": "postgres",                                                                            │
│   "fecha_ultima_modificacion": null,                                                                            │
│   "tipo_carga_proceso": "Carga Incremental",                                                                    │
│   "nombre_proceso": "Descargar datos procesos judiciales",                                                      │
│   "fecha_fin_extraccion": null,                                                                                 │
│   "observaciones": null,                                                                                        │
│   "estado_registro": "Activo",                                                                                  │
│   "fecha_creacion": "2026-01-31 12:14:24"                                                                       │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────── Parámetros ───────────────────────────╮
│ {                                                                │
│   "tabla_destino_p1": "simulacion.procesos_judiciales_colombia", │
│   "url_fuente_p1": "www.datos.gov.co",                           │
│   "limite_fuente_p1": "15365",                                   │
│   "id_fuente_p1": "52tq-ag6c"                                    │
│ }                                                                │
╰──────────────────────────────────────────────────────────────────╯

**Paso 5.** Simular el proceso ETL.
Este proceso ejecuta la copia (descarga y almacenamiento) de los datos de los proceso judiciales de Colombia desde la fuente (portal de datos abiertos de Colombia).
Este proceso aprovecha las funcionalidades del *Piloto de Framework* así:
* Hace uso de la gestión de **parámetros** para personalizar la descarga de datos. Valores requeridos por la fuente de datos como el *identificador del conjunto de datos* a consultar o la *cantidad de registros* a descargar se gestionan desde el *Repositorio de datos" en lugar de escribirlos literalmente en el código.
* Hace uso de la gestión de **procesos** para controlar las características básicas del proceso. Valores como el *tipo de carga* que ejecutará el proceso ETL o el *correo electrónico* para notificaciones se gestionan desde el "Repositorio de datos" en lugar de escribirlos literalmente en el código.

Al finalizar la ejecución del proceso ETL, el **Piloto de Framework** permite registrar los datos de particulares de la ejecución, tales como el *momento de inicio y fin* del proceso, el *estado de la ejecución* o la *cantidad de registros procesados*

In [5]:
inicio_ejecucion = datetime.now()
mensaje = None
if framework.proceso._error_precedencias:
    framework.registro_log(
            inicio_ejecucion,
            EstadoEjecucion.ERROR,
            "Error de ejecución por precedencias: verifique el estado de la última ejecución de los procesos precedentes.",
            0
        )
if framework.proceso._ejecutar:
    try:
        par_url_fuente = framework.parametros.get("url_fuente_p1",'')
        par_id_fuente = framework.parametros.get("id_fuente_p1",'')
        par_limite_fuente = framework.parametros.get("limite_fuente_p1",'')
        par_tabla_destino = framework.parametros.get("tabla_destino_p1",'')
        cliente = Socrata("{0}".format(par_url_fuente), None)
        conteo = framework.bd_sesion.exec(sqlalchemy.text(f"select count(1) from {framework.parametros.get("tabla_destino_p1",'')}"))
        secuencia = conteo.fetchall()[0][0]
        fuente = cliente.get("{0}".format(par_id_fuente), limit=par_limite_fuente,offset=secuencia)
        conjunto_datos = pd.DataFrame.from_records(fuente)
        grupo_1 = re.search(r'(?<=[.])\w+', par_tabla_destino)
        grupo_2 = re.search(r'(?<![.])\w+', par_tabla_destino)
        tabla: str | None = None
        esquema: str | None = None
        if grupo_1:
            tabla = grupo_1.group(0)
            esquema = grupo_2.group(0)
        elif grupo_2:
            tabla = grupo_2.group(0)
        else:
            tabla = ''
        tipo_carga = 'append' if framework.proceso.tipo_carga_proceso == TipoCarga.INC else 'replace'
        conjunto_datos.to_sql(
            name=tabla,
            schema = esquema,
            con=framework.bd_motor,
            if_exists=tipo_carga,
            index=False
        )
        estado_proceso = EstadoEjecucion.CORRECTO
        registros_procesados = len(conjunto_datos)
    except (Exception) as error:
        estado_proceso = EstadoEjecucion.ERROR
        mensaje = f"Error en la ejecución: {error}"
        registros_procesados = 0
    finally:
        framework.registro_log(
            inicio_ejecucion,
            estado_proceso,
            mensaje,
            registros_procesados
        )

╭─────────────────────────────────────────────── Registro de log ───────────────────────────────────────────────╮
│ {                                                                                                             │
│   "id_ejecucion": "603b6f86-991d-4753-9f1d-610ea3da0a11",                                                     │
│   "id_proceso": "a0b51bd8-ad38-4cf1-b587-eb2ac1d9cf34",                                                       │
│   "fecha_inicio_ejecucion": "2026-02-02 18:38:02",                                                            │
│   "fecha_fin_ejecucion": "2026-02-02 18:38:18",                                                               │
│   "estado": "Error",                                                                                          │
│   "mensaje": "Error en la ejecución: HTTPSConnectionPool(host='www.datos.gov.co', port=443): Read timed out." │
│ }                                                                                                             │
╰───────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

(Denied) Denied by the resource provider.
Code: Denied
Message: Denied by the resource provider.

**Paso 6.** Limpieza

In [6]:
import gc as _gc
for name in dir():
    if not name.startswith('_'):
        del globals()[name]
del name
_gc.collect()


841